In [1]:
import re
import time
import requests
import pandas as pd
from pathlib import Path
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
import warnings
from google.colab import files

warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

In [2]:
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Uploaded: []


In [3]:
remaining = pd.read_csv("remaining_52_files_rebuilt.csv")
events_clean = pd.read_csv("events_clean.csv")

for df in [remaining, events_clean]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()

remaining["filing_date"] = pd.to_datetime(remaining["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")
events_clean["filing_date"] = pd.to_datetime(events_clean["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

repair_df = remaining.merge(
    events_clean[["ticker", "cik", "filing_date", "filing_type", "accession_number"]],
    on=["ticker", "filing_date", "filing_type", "accession_number"],
    how="left"
)

print("Rows after merge:", len(repair_df))
print("Missing cik:", repair_df["cik"].isna().sum())
display(repair_df.head())

Rows after merge: 52
Missing cik: 0


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes,cik
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_20191031_10-K_000032019319000119.txt,1018,0.015798,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,320193
1,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1800
2,ABT,2024-02-16,10-K,0001628280-24-005348,ABT_20240216_10-K_000162828024005348.txt,733,0.059546,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1800
3,CMCSA,2022-02-02,10-K,0001166691-22-000009,CMCSA_20220202_10-K_000116669122000009.txt,1249,0.039080,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1166691
4,CMCSA,2023-02-03,10-K,0001166691-23-000010,CMCSA_20230203_10-K_000116669123000010.txt,1231,0.036038,False,True,True,...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN,1166691


In [4]:
ticker_counts = repair_df["ticker"].value_counts().reset_index()
ticker_counts.columns = ["ticker", "count"]
display(ticker_counts)

,ticker,count
0,USB,14
1,JCI,13
2,SCHW,6
3,KLAC,5
4,PH,3
5,CMCSA,3
6,CRWD,3
7,ABT,2
8,AAPL,1
9,COF,1


In [5]:
BASE_DIR = Path("/content/family_rescue")
OUT_DIR = BASE_DIR / "repaired_txt"
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPAIR_LOG_CSV = BASE_DIR / "family_rescue_log.csv"
REVIEW_CSV = BASE_DIR / "family_rescue_review.csv"

In [6]:
HEADERS = {
    "User-Agent": "Cardiff University Student abhishekjc23@gmail.com",
    "Accept-Encoding": "gzip, deflate",
    "Host": "www.sec.gov"
}

def cik_nolead(cik):
    if pd.isna(cik):
        return None
    return str(int(float(cik)))

def acc_nodash(acc):
    return str(acc).replace("-", "")

def filing_base_url(cik, accession_number):
    return f"https://www.sec.gov/Archives/edgar/data/{cik_nolead(cik)}/{acc_nodash(accession_number)}"

def index_json_url(cik, accession_number):
    return filing_base_url(cik, accession_number) + "/index.json"

def get_index_json(cik, accession_number):
    url = index_json_url(cik, accession_number)
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.json()

def safe_int(value, default=0):
    try:
        value = str(value).strip()
        if value == "" or value.lower() == "none":
            return default
        return int(value)
    except Exception:
        return default

def choose_primary_doc(index_json, filing_type):
    try:
        items = index_json["directory"]["item"]
    except Exception:
        return None

    docs = []
    for x in items:
        name = str(x.get("name", ""))
        lower = name.lower()
        if lower.endswith((".htm", ".html", ".txt")):
            docs.append(x)

    if not docs:
        return None

    bad_words = ["ex-", "exhibit", "xbrl", "xml", "graphic", "image", "zip", "def14a", "8-k"]
    filtered = []
    for d in docs:
        nm = str(d.get("name", "")).lower()
        if any(b in nm for b in bad_words):
            continue
        filtered.append(d)

    if not filtered:
        filtered = docs

    ft = filing_type.lower().replace("-", "").replace("_", "")
    filing_pref = []
    for d in filtered:
        nm = str(d.get("name", "")).lower().replace("-", "").replace("_", "")
        if ft in nm:
            filing_pref.append(d)
        elif filing_type == "10-K" and "10k" in nm:
            filing_pref.append(d)
        elif filing_type == "10-Q" and "10q" in nm:
            filing_pref.append(d)

    if filing_pref:
        filtered = filing_pref

    filtered = sorted(filtered, key=lambda x: safe_int(x.get("size", 0), 0), reverse=True)
    return filtered[0]["name"] if filtered else None

def download_filing_doc(cik, accession_number, filename):
    url = filing_base_url(cik, accession_number) + f"/{filename}"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    return r.text

def html_to_text(html):
    soup = BeautifulSoup(html, "lxml")
    for tag in soup(["script", "style", "ix:header", "header", "footer"]):
        tag.decompose()
    text = soup.get_text("\n")
    text = text.replace("\xa0", " ")
    text = text.replace("’", "'")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n\n", text)
    return text.strip()

In [7]:
def preview_unresolved_row(i, start_chars=2500, end_chars=1200):
    row = repair_df.iloc[i]
    idx = get_index_json(row["cik"], row["accession_number"])
    primary_doc = choose_primary_doc(idx, row["filing_type"])
    html = download_filing_doc(row["cik"], row["accession_number"], primary_doc)
    text = html_to_text(html)

    print("=" * 120)
    print("ROW:", i)
    print("TICKER:", row["ticker"])
    print("DATE:", row["filing_date"])
    print("TYPE:", row["filing_type"])
    print("PRIMARY DOC:", primary_doc)
    print("=" * 120)
    print("\n--- START OF FILING TEXT ---\n")
    print(text[:start_chars])
    print("\n--- END OF FILING TEXT ---\n")
    print(text[-end_chars:])
    print("\nWord count:", len(re.findall(r"\b[a-zA-Z]+\b", text)))
    print("=" * 120)

In [8]:
preview_unresolved_row(0)

ROW: 0
TICKER: AAPL
DATE: 2019-10-31
TYPE: 10-K
PRIMARY DOC: a10-k20199282019.htm

--- START OF FILING TEXT ---

Document

 
UNITED STATES
SECURITIES AND EXCHANGE COMMISSION
Washington, D.C. 20549
 
FORM 
10-K
 
 
(Mark One)
☒
 
ANNUAL REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the fiscal year ended 
September 28, 2019
or
☐
 
TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934
For the transition period from 
 
 to 
 
.
Commission File Number: 
001-36743
 
Apple Inc.
(Exact name of Registrant as specified in its charter)
 
California
 
94-2404110
(State or other jurisdiction
of incorporation or organization)
 
(I.R.S. Employer Identification No.)
 
 
 
 
 
One Apple Park Way
 
 
Cupertino
 
California
 
95014
(Address of principal executive offices)
 
(Zip Code)
(
408
) 
996-1010
(Registrant's telephone number, including area code)
 
Securities registered pursuant to Section 12(b) of the Act:
Title of each class

In [9]:
def show_ticker_rows(ticker):
    sub = repair_df.loc[repair_df["ticker"] == ticker, ["ticker", "filing_date", "filing_type", "accession_number"]].copy()
    print("Rows for", ticker, "=", len(sub))
    display(sub.reset_index())

In [10]:
show_ticker_rows("AAPL")
show_ticker_rows("ABT")
show_ticker_rows("CMCSA")

Rows for AAPL = 1


,index,ticker,filing_date,filing_type,accession_number
0,0,AAPL,2019-10-31,10-K,0000320193-19-000119


Rows for ABT = 2


,index,ticker,filing_date,filing_type,accession_number
0,1,ABT,2023-02-17,10-K,0001628280-23-004026
1,2,ABT,2024-02-16,10-K,0001628280-24-005348


Rows for CMCSA = 3


,index,ticker,filing_date,filing_type,accession_number
0,3,CMCSA,2022-02-02,10-K,0001166691-22-000009
1,4,CMCSA,2023-02-03,10-K,0001166691-23-000010
2,5,CMCSA,2024-01-31,10-K,0001166691-24-000011


In [11]:
FAMILY_RULES = {
    # Examples - edit as needed after previewing one filing per ticker
    "AAPL": {
        "10-K": {
            "start_patterns": [
                r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*8[\.\-:\s]+financial statements",
            ],
        }
    },
    "ABT": {
        "10-K": {
            "start_patterns": [
                r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*8[\.\-:\s]+financial statements",
            ],
        }
    },
    "CMCSA": {
        "10-K": {
            "start_patterns": [
                r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*8[\.\-:\s]+financial statements",
            ],
        }
    },
    "KLAC": {
        "10-K": {
            "start_patterns": [
                r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*8[\.\-:\s]+financial statements",
            ],
        }
    },
}

In [12]:
def extract_with_family_rule(text, ticker, filing_type):
    text = text.replace("\xa0", " ")
    text = text.replace("’", "'")
    low = text.lower()

    rules = FAMILY_RULES.get(ticker, {}).get(filing_type, None)
    if rules is None:
        return None, "NO_RULE"

    starts = []
    for pat in rules["start_patterns"]:
        for m in re.finditer(pat, low, flags=re.I):
            starts.append(m.start())

    ends = []
    for pat in rules["end_patterns"]:
        for m in re.finditer(pat, low, flags=re.I):
            ends.append(m.start())

    starts = sorted(set(starts))
    ends = sorted(set(ends))

    if not starts:
        return None, "NO_START"
    if not ends:
        return None, "NO_END"

    # Prefer later starts to avoid TOC/reference hits
    doc_len = len(text)
    starts = [s for s in starts if s > doc_len * 0.10] or starts

    best_pair = None
    best_span = None

    for s in starts:
        valid_ends = [e for e in ends if e > s]
        if not valid_ends:
            continue

        e = valid_ends[0]
        span = e - s

        if 3000 <= span <= 400000:
            best_pair = (s, e)
            best_span = span
            break

        if best_pair is None:
            best_pair = (s, e)
            best_span = span

    if best_pair is None:
        return None, "NO_VALID_SPAN"

    s, e = best_pair
    mda = text[s:e].strip()

    wc = len(re.findall(r"\b[a-zA-Z]+\b", mda))
    if wc < 800:
        return None, "TOO_SHORT"

    first_500 = re.sub(r"\s+", " ", mda[:500]).lower()
    if filing_type == "10-K":
        good = ("item 7" in first_500 and "management's discussion" in first_500)
    else:
        good = ("item 2" in first_500 and "management's discussion" in first_500)

    if not good:
        return None, "BAD_START"

    if (
        "table of contents" in first_500 or
        "under the heading" in first_500 or
        "refer to item" in first_500 or
        "see item" in first_500
    ):
        return None, "BAD_START"

    return mda, "OK"

In [13]:
def repair_one_family_rule(row, sleep_seconds=0.2):
    ticker = row["ticker"]
    cik = row["cik"]
    filing_date = row["filing_date"]
    filing_type = row["filing_type"]
    accession_number = row["accession_number"]

    out = {
        "ticker": ticker,
        "filing_date": filing_date,
        "filing_type": filing_type,
        "accession_number": accession_number,
        "repair_status": "",
        "primary_doc": "",
        "repaired_filename": ""
    }

    try:
        if pd.isna(cik):
            out["repair_status"] = "NO_CIK"
            return out

        idx = get_index_json(cik, accession_number)
        primary_doc = choose_primary_doc(idx, filing_type)
        if primary_doc is None:
            out["repair_status"] = "NO_PRIMARY_DOC"
            return out

        out["primary_doc"] = primary_doc

        html = download_filing_doc(cik, accession_number, primary_doc)
        text = html_to_text(html)

        mda_text, status = extract_with_family_rule(text, ticker, filing_type)

        if status != "OK":
            out["repair_status"] = status
            return out

        fname = f"{ticker}_{filing_date}_{filing_type}_{accession_number}_FAMILY52.txt"
        with open(OUT_DIR / fname, "w", encoding="utf-8") as f:
            f.write(mda_text)

        out["repair_status"] = "OK"
        out["repaired_filename"] = fname

        time.sleep(sleep_seconds)
        return out

    except Exception as e:
        out["repair_status"] = f"ERROR: {str(e)[:120]}"
        return out

In [14]:
TARGET_TICKERS = ["AAPL", "ABT", "CMCSA", "KLAC"]

target_df = repair_df.loc[repair_df["ticker"].isin(TARGET_TICKERS)].copy()
print("Target rows:", len(target_df))
display(target_df[["ticker", "filing_date", "filing_type", "accession_number"]].head(20))

Target rows: 11


,ticker,filing_date,filing_type,accession_number
0,AAPL,2019-10-31,10-K,0000320193-19-000119
1,ABT,2023-02-17,10-K,0001628280-23-004026
2,ABT,2024-02-16,10-K,0001628280-24-005348
3,CMCSA,2022-02-02,10-K,0001166691-22-000009
4,CMCSA,2023-02-03,10-K,0001166691-23-000010
5,CMCSA,2024-01-31,10-K,0001166691-24-000011
23,KLAC,2019-08-16,10-K,0000319201-19-000031
24,KLAC,2021-08-06,10-K,0000319201-21-000029
25,KLAC,2022-04-29,10-Q,0000319201-22-000012
26,KLAC,2022-08-05,10-K,0000319201-22-000023


In [15]:
results = []

for i, row in target_df.iterrows():
    result = repair_one_family_rule(row)
    results.append(result)

    if len(results) % 10 == 0:
        print(f"Processed {len(results)}/{len(target_df)}")

repair_log = pd.DataFrame(results)
repair_log.to_csv(REPAIR_LOG_CSV, index=False)

print("Family repair complete.")
display(repair_log["repair_status"].value_counts())
display(repair_log.head())

Processed 10/11
Family repair complete.


,count
repair_status,
OK,3
TOO_SHORT,3
NO_START,2
BAD_START,2
NO_RULE,1


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,OK,a10-k20199282019.htm,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...
1,ABT,2023-02-17,10-K,0001628280-23-004026,NO_START,abt-20221231x10kexx21.htm,
2,ABT,2024-02-16,10-K,0001628280-24-005348,NO_START,abt-20231231x10kexx312.htm,
3,CMCSA,2022-02-02,10-K,0001166691-22-000009,BAD_START,cmcsa-20211231.htm,
4,CMCSA,2023-02-03,10-K,0001166691-23-000010,BAD_START,cmcsa-20221231.htm,


In [16]:
def read_text_safe(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

review_rows = []

for _, row in repair_log.loc[repair_log["repair_status"] == "OK"].iterrows():
    path = OUT_DIR / row["repaired_filename"]
    text = read_text_safe(path)

    review_rows.append({
        "ticker": row["ticker"],
        "filing_date": row["filing_date"],
        "filing_type": row["filing_type"],
        "accession_number": row["accession_number"],
        "repaired_filename": row["repaired_filename"],
        "word_count": len(re.findall(r"\b[a-zA-Z]+\b", text)),
        "first_300_chars": text[:300].replace("\n", " "),
        "first_1000_chars": text[:1000].replace("\n", " "),
    })

review_df = pd.DataFrame(review_rows)
review_df.to_csv(REVIEW_CSV, index=False)

print("Saved review CSV:", REVIEW_CSV)
display(review_df)

Saved review CSV: /content/family_rescue/family_rescue_review.csv


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,first_300_chars,first_1000_chars
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,4715,Item 7. Management's Discussion and Analysis o...,Item 7. Management's Discussion and Analysis o...
1,CMCSA,2024-01-31,10-K,0001166691-24-000011,CMCSA_2024-01-31_10-K_0001166691-24-000011_FAM...,13100,Item 7: Management's Discussion and Analysis o...,Item 7: Management's Discussion and Analysis o...
2,KLAC,2023-08-04,10-K,0000319201-23-000031,KLAC_2023-08-04_10-K_0000319201-23-000031_FAMI...,10621,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...


In [17]:
ok_family = repair_log.loc[
    repair_log["repair_status"] == "OK",
    ["ticker", "filing_date", "filing_type", "accession_number", "repaired_filename"]
].copy()

print("Family OK files:", len(ok_family))
display(ok_family)

Family OK files: 3


,ticker,filing_date,filing_type,accession_number,repaired_filename
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...
5,CMCSA,2024-01-31,10-K,0001166691-24-000011,CMCSA_2024-01-31_10-K_0001166691-24-000011_FAM...
10,KLAC,2023-08-04,10-K,0000319201-23-000031,KLAC_2023-08-04_10-K_0000319201-23-000031_FAMI...


In [18]:
def preview_repaired_file(filename, start_chars=2500, end_chars=1200):
    path = OUT_DIR / filename
    if not path.exists():
        print("File not found:", path)
        return

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    print("=" * 120)
    print("FILE:", filename)
    print("=" * 120)
    print("\n--- START PREVIEW ---\n")
    print(text[:start_chars])
    print("\n--- END PREVIEW ---\n")
    print(text[-end_chars:])
    print("\nWord count:", len(re.findall(r"\b[a-zA-Z]+\b", text)))
    print("=" * 120)

In [19]:
for i in range(len(ok_family)):
    fname = ok_family.iloc[i]["repaired_filename"]
    print(f"\n### Reviewing row {i}")
    preview_repaired_file(fname)


### Reviewing row 0
FILE: AAPL_2019-10-31_10-K_0000320193-19-000119_FAMILY52.txt

--- START PREVIEW ---

Item 7.
Management's Discussion and Analysis of Financial Condition and Results of Operations
This section and other parts of this Annual Report on Form 10-K (“Form 10-K”) contain forward-looking statements, within the meaning of the Private Securities Litigation Reform Act of 1995, that involve risks and uncertainties. Forward-looking statements provide current expectations of future events based on certain assumptions and include any statement that does not directly relate to any historical or current fact. Forward-looking statements can also be identified by words such as “future,” “anticipates,” “believes,” “estimates,” “expects,” “intends,” “plans,” “predicts,” “will,” “would,” “could,” “can,” “may,” and similar terms. Forward-looking statements are not guarantees of future performance and the Company's actual results may differ significantly from the results discussed in the 

In [20]:
ok_family = pd.read_csv("remaining52_ok_4_files_rich_review.csv") if False else ok_family.copy()

correct_keys_2 = {
    ("AAPL", "2019-10-31", "10-K"),
    ("KLAC", "2023-08-04", "10-K"),
}

ok_family["ticker"] = ok_family["ticker"].astype(str).str.strip().str.upper()
ok_family["filing_type"] = ok_family["filing_type"].astype(str).str.strip().str.upper()
ok_family["accession_number"] = ok_family["accession_number"].astype(str).str.strip()
ok_family["filing_date"] = pd.to_datetime(ok_family["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

ok_family["manual_label"] = ""
ok_family["notes"] = ""

ok_family.loc[
    ok_family.apply(lambda r: (r["ticker"], r["filing_date"], r["filing_type"]) in correct_keys_2, axis=1),
    ["manual_label", "notes"]
] = ["correct", "accepted after manual preview"]

correct_2 = ok_family.loc[ok_family["manual_label"] == "correct"].copy()
correct_2.to_csv("/content/correct_2_files.csv", index=False)

print("Correct rows:", len(correct_2))
display(correct_2)

Correct rows: 2


,ticker,filing_date,filing_type,accession_number,repaired_filename,manual_label,notes
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,correct,accepted after manual preview
10,KLAC,2023-08-04,10-K,0000319201-23-000031,KLAC_2023-08-04_10-K_0000319201-23-000031_FAMI...,correct,accepted after manual preview


In [21]:
remaining_52 = pd.read_csv("remaining_52_files_rebuilt.csv")
correct_2 = pd.read_csv("/content/correct_2_files.csv")

for df in [remaining_52, correct_2]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

correct_2_keys = set(
    zip(
        correct_2["ticker"],
        correct_2["filing_date"],
        correct_2["filing_type"],
        correct_2["accession_number"]
    )
)

remaining_42 = remaining_52.loc[
    ~remaining_52.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in correct_2_keys,
        axis=1
    )
].copy()

remaining_42.to_csv("/content/remaining_42_files.csv", index=False)

print("Remaining unresolved files:", len(remaining_42))
display(remaining_42.head())

Remaining unresolved files: 50


,ticker,filing_date,filing_type,accession_number,mda_filename,word_count_actual,digit_ratio_full,good_start_detected,suspicious_start_detected,contains_market_risk_start,...,first_400_chars,needs_manual_review,review_reason,manual_label,notes,notes_final,strict_label,strict_notes,original_manual_label,original_notes
1,ABT,2023-02-17,10-K,0001628280-23-004026,ABT_20230217_10-K_000162828023004026.txt,723,0.060734,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
2,ABT,2024-02-16,10-K,0001628280-24-005348,ABT_20240216_10-K_000162828024005348.txt,733,0.059546,False,True,True,...,A. QUANTITATIVE AND QUALITATIVE DISCLOSURES AB...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
3,CMCSA,2022-02-02,10-K,0001166691-22-000009,CMCSA_20220202_10-K_000116669122000009.txt,1249,0.039080,False,True,True,...,A: Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
4,CMCSA,2023-02-03,10-K,0001166691-23-000010,CMCSA_20230203_10-K_000116669123000010.txt,1231,0.036038,False,True,True,...,A: Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN
5,CMCSA,2024-01-31,10-K,0001166691-24-000011,CMCSA_20240131_10-K_000116669124000011.txt,1139,0.033118,False,True,True,...,A: Quantitative and Qualitative Disclosures Ab...,True,flagged,incorrect,NaN,NaN,incorrect,NaN,incorrect,NaN


In [22]:
target_df = repair_df.copy()

print("Rows to process:", len(target_df))
display(target_df[["ticker", "filing_date", "filing_type", "accession_number"]].head(20))

Rows to process: 52


,ticker,filing_date,filing_type,accession_number
0,AAPL,2019-10-31,10-K,0000320193-19-000119
1,ABT,2023-02-17,10-K,0001628280-23-004026
2,ABT,2024-02-16,10-K,0001628280-24-005348
3,CMCSA,2022-02-02,10-K,0001166691-22-000009
4,CMCSA,2023-02-03,10-K,0001166691-23-000010
5,CMCSA,2024-01-31,10-K,0001166691-24-000011
6,COF,2022-02-25,10-K,0000927628-22-000106
7,CRWD,2022-03-16,10-K,0001535527-22-000006
8,CRWD,2023-03-09,10-K,0001535527-23-000008
9,CRWD,2024-03-07,10-K,0001535527-24-000007


In [23]:
ticker_counts = repair_df["ticker"].value_counts().reset_index()
ticker_counts.columns = ["ticker", "count"]
display(ticker_counts)

,ticker,count
0,USB,14
1,JCI,13
2,SCHW,6
3,KLAC,5
4,PH,3
5,CMCSA,3
6,CRWD,3
7,ABT,2
8,AAPL,1
9,COF,1


In [24]:
show_ticker_rows("USB")
show_ticker_rows("JCI")
show_ticker_rows("SCHW")
show_ticker_rows("KLAC")

Rows for USB = 14


,index,ticker,filing_date,filing_type,accession_number
0,38,USB,2020-05-07,10-Q,0001193125-20-136359
1,39,USB,2020-08-06,10-Q,0001193125-20-211979
2,40,USB,2020-11-05,10-Q,0001193125-20-286983
3,41,USB,2021-05-04,10-Q,0001193125-21-150169
4,42,USB,2021-08-03,10-Q,0001193125-21-234982
5,43,USB,2021-11-02,10-Q,0001193125-21-317037
6,44,USB,2022-05-03,10-Q,0001193125-22-138788
7,45,USB,2022-08-04,10-Q,0001193125-22-212040
8,46,USB,2022-11-01,10-Q,0001193125-22-275036
9,47,USB,2023-05-08,10-Q,0001193125-23-137980


Rows for JCI = 13


,index,ticker,filing_date,filing_type,accession_number
0,10,JCI,2020-07-31,10-Q,0000833444-20-000036
1,11,JCI,2021-01-29,10-Q,0000833444-21-000011
2,12,JCI,2021-04-30,10-Q,0000833444-21-000020
3,13,JCI,2021-07-30,10-Q,0000833444-21-000040
4,14,JCI,2022-02-02,10-Q,0000833444-22-000005
5,15,JCI,2022-05-04,10-Q,0000833444-22-000017
6,16,JCI,2022-08-04,10-Q,0000833444-22-000036
7,17,JCI,2023-02-01,10-Q,0000833444-23-000005
8,18,JCI,2023-05-05,10-Q,0000833444-23-000014
9,19,JCI,2023-08-02,10-Q,0000833444-23-000030


Rows for SCHW = 6


,index,ticker,filing_date,filing_type,accession_number
0,32,SCHW,2021-05-07,10-Q,0000316709-21-000029
1,33,SCHW,2022-05-09,10-Q,0000316709-22-000018
2,34,SCHW,2022-11-08,10-Q,0000316709-22-000045
3,35,SCHW,2024-05-09,10-Q,0000316709-24-000044
4,36,SCHW,2024-08-08,10-Q,0000316709-24-000060
5,37,SCHW,2024-11-08,10-Q,0000316709-24-000069


Rows for KLAC = 5


,index,ticker,filing_date,filing_type,accession_number
0,23,KLAC,2019-08-16,10-K,0000319201-19-000031
1,24,KLAC,2021-08-06,10-K,0000319201-21-000029
2,25,KLAC,2022-04-29,10-Q,0000319201-22-000012
3,26,KLAC,2022-08-05,10-K,0000319201-22-000023
4,27,KLAC,2023-08-04,10-K,0000319201-23-000031


In [25]:
FAMILY_RULES = {
    "USB": {
        "10-Q": {
            "start_patterns": [
                r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*4[\.\-:\s]+controls and procedures",
                r"^item\s*3\b",
                r"^item\s*4\b",
            ],
        }
    },

    "JCI": {
        "10-Q": {
            "start_patterns": [
                r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*4[\.\-:\s]+controls and procedures",
                r"^item\s*3\b",
                r"^item\s*4\b",
            ],
        }
    },

    "SCHW": {
        "10-Q": {
            "start_patterns": [
                r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*4[\.\-:\s]+controls and procedures",
                r"^item\s*3\b",
                r"^item\s*4\b",
            ],
        }
    },

    "KLAC": {
        "10-K": {
            "start_patterns": [
                r"item\s*7[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*7a[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*8[\.\-:\s]+financial statements",
                r"^item\s*8\b",
            ],
        },
        "10-Q": {
            "start_patterns": [
                r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis of financial condition and results of operations",
            ],
            "end_patterns": [
                r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*4[\.\-:\s]+controls and procedures",
                r"^item\s*3\b",
                r"^item\s*4\b",
            ],
        }
    }
}

In [26]:
TARGET_TICKERS = ["USB", "JCI", "SCHW", "KLAC"]

target_df = repair_df.loc[repair_df["ticker"].isin(TARGET_TICKERS)].copy()

print("Target rows:", len(target_df))
display(target_df[["ticker", "filing_date", "filing_type", "accession_number"]])

Target rows: 38


,ticker,filing_date,filing_type,accession_number
10,JCI,2020-07-31,10-Q,0000833444-20-000036
11,JCI,2021-01-29,10-Q,0000833444-21-000011
12,JCI,2021-04-30,10-Q,0000833444-21-000020
13,JCI,2021-07-30,10-Q,0000833444-21-000040
14,JCI,2022-02-02,10-Q,0000833444-22-000005
15,JCI,2022-05-04,10-Q,0000833444-22-000017
16,JCI,2022-08-04,10-Q,0000833444-22-000036
17,JCI,2023-02-01,10-Q,0000833444-23-000005
18,JCI,2023-05-05,10-Q,0000833444-23-000014
19,JCI,2023-08-02,10-Q,0000833444-23-000030


In [27]:
results = []

for i, row in target_df.iterrows():
    result = repair_one_family_rule(row)
    results.append(result)

    if len(results) % 10 == 0 or len(results) == len(target_df):
        print(f"Processed {len(results)}/{len(target_df)}")

repair_log = pd.DataFrame(results)
repair_log.to_csv(REPAIR_LOG_CSV, index=False)

print("Family repair complete.")
display(repair_log["repair_status"].value_counts())
display(repair_log.head())

Processed 10/38
Processed 20/38
Processed 30/38
Processed 38/38
Family repair complete.


,count
repair_status,
NO_START,20
NO_END,14
BAD_START,3
OK,1


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename
0,JCI,2020-07-31,10-Q,0000833444-20-000036,NO_START,q3ex311fy2010-q.htm,
1,JCI,2021-01-29,10-Q,0000833444-21-000011,NO_START,q1ex101fy2110-q.htm,
2,JCI,2021-04-30,10-Q,0000833444-21-000020,NO_START,q2ex104fy2110-q.htm,
3,JCI,2021-07-30,10-Q,0000833444-21-000040,NO_START,q3ex311fy2110-q.htm,
4,JCI,2022-02-02,10-Q,0000833444-22-000005,NO_START,q1ex101fy2210-q.htm,


In [28]:
TARGET_TICKERS = ["USB"]

target_df = repair_df.loc[repair_df["ticker"].isin(TARGET_TICKERS)].copy()

print("Target rows:", len(target_df))
display(target_df[["ticker", "filing_date", "filing_type", "accession_number"]])

Target rows: 14


,ticker,filing_date,filing_type,accession_number
38,USB,2020-05-07,10-Q,0001193125-20-136359
39,USB,2020-08-06,10-Q,0001193125-20-211979
40,USB,2020-11-05,10-Q,0001193125-20-286983
41,USB,2021-05-04,10-Q,0001193125-21-150169
42,USB,2021-08-03,10-Q,0001193125-21-234982
43,USB,2021-11-02,10-Q,0001193125-21-317037
44,USB,2022-05-03,10-Q,0001193125-22-138788
45,USB,2022-08-04,10-Q,0001193125-22-212040
46,USB,2022-11-01,10-Q,0001193125-22-275036
47,USB,2023-05-08,10-Q,0001193125-23-137980


In [29]:
FAMILY_RULES = {
    "USB": {
        "10-Q": {
            "start_patterns": [
                r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis of financial condition and results of operations",
                r"management[' ]s discussion and analysis",
            ],
            "end_patterns": [
                r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
                r"item\s*4[\.\-:\s]+controls and procedures",
                r"^item\s*3\b",
                r"^item\s*4\b",
            ],
        }
    }
}

In [30]:
results = []

for i, row in target_df.iterrows():
    result = repair_one_family_rule(row)
    results.append(result)

    if len(results) % 10 == 0 or len(results) == len(target_df):
        print(f"Processed {len(results)}/{len(target_df)}")

repair_log = pd.DataFrame(results)
repair_log.to_csv(REPAIR_LOG_CSV, index=False)

print("USB family repair complete.")
display(repair_log["repair_status"].value_counts())
display(repair_log.head())

Processed 10/14
Processed 14/14
USB family repair complete.


,count
repair_status,
NO_END,14


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename
0,USB,2020-05-07,10-Q,0001193125-20-136359,NO_END,d897119d10q.htm,
1,USB,2020-08-06,10-Q,0001193125-20-211979,NO_END,d890129d10q.htm,
2,USB,2020-11-05,10-Q,0001193125-20-286983,NO_END,d947218d10q.htm,
3,USB,2021-05-04,10-Q,0001193125-21-150169,NO_END,d137164d10q.htm,
4,USB,2021-08-03,10-Q,0001193125-21-234982,NO_END,d913559d10q.htm,


In [31]:
ok_usb = repair_log.loc[repair_log["repair_status"] == "OK"].copy()
print("Recovered OK USB files:", len(ok_usb))
display(ok_usb)


Recovered OK USB files: 0


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename


In [32]:
def extract_usb_fallback_end(text):
    text = text.replace("\xa0", " ")
    text = text.replace("’", "'")
    low = text.lower()

    # start patterns
    start_patterns = [
        r"item\s*2[\.\-:\s]+management[' ]s discussion and analysis of financial condition and results of operations",
        r"management[' ]s discussion and analysis of financial condition and results of operations",
        r"management[' ]s discussion and analysis",
    ]

    starts = []
    for pat in start_patterns:
        for m in re.finditer(pat, low, flags=re.I):
            starts.append(m.start())

    starts = sorted(set(starts))
    if not starts:
        return None, "NO_START"

    doc_len = len(text)

    # prefer later starts to avoid TOC/reference hits
    starts = [s for s in starts if s > doc_len * 0.10] or starts

    # end patterns
    end_patterns = [
        r"item\s*3[\.\-:\s]+quantitative and qualitative disclosures about market risk",
        r"item\s*4[\.\-:\s]+controls and procedures",
        r"^item\s*3\b",
        r"^item\s*4\b",
    ]

    best_pair = None

    for s in starts:
        # first try explicit end patterns after s
        end_hits = []
        for pat in end_patterns:
            for m in re.finditer(pat, low[s:], flags=re.I | re.M):
                end_hits.append(s + m.start())

        end_hits = sorted(set(end_hits))

        # if no explicit end, fallback to next major item line
        if not end_hits:
            fallback = re.search(r"(?im)^item\s*(3|4)\b", low[s:])
            if fallback:
                end_hits = [s + fallback.start()]

        if not end_hits:
            continue

        e = end_hits[0]
        span = e - s

        if 3000 <= span <= 400000:
            best_pair = (s, e)
            break

    if best_pair is None:
        return None, "NO_END"

    s, e = best_pair
    mda = text[s:e].strip()

    wc = len(re.findall(r"\b[a-zA-Z]+\b", mda))
    if wc < 800:
        return None, "TOO_SHORT"

    first_500 = re.sub(r"\s+", " ", mda[:500]).lower()

    good = (
        ("item 2" in first_500 and "management's discussion" in first_500) or
        ("management's discussion and analysis" in first_500 and "financial condition" in first_500)
    )

    if not good:
        return None, "BAD_START"

    if (
        "table of contents" in first_500 or
        "under the heading" in first_500 or
        "refer to item" in first_500 or
        "see item" in first_500
    ):
        return None, "BAD_START"

    return mda, "OK"

In [33]:
def repair_one_usb_fallback(row, sleep_seconds=0.2):
    ticker = row["ticker"]
    cik = row["cik"]
    filing_date = row["filing_date"]
    filing_type = row["filing_type"]
    accession_number = row["accession_number"]

    out = {
        "ticker": ticker,
        "filing_date": filing_date,
        "filing_type": filing_type,
        "accession_number": accession_number,
        "repair_status": "",
        "primary_doc": "",
        "repaired_filename": ""
    }

    try:
        if pd.isna(cik):
            out["repair_status"] = "NO_CIK"
            return out

        idx = get_index_json(cik, accession_number)
        primary_doc = choose_primary_doc(idx, filing_type)
        if primary_doc is None:
            out["repair_status"] = "NO_PRIMARY_DOC"
            return out

        out["primary_doc"] = primary_doc

        html = download_filing_doc(cik, accession_number, primary_doc)
        text = html_to_text(html)

        mda_text, status = extract_usb_fallback_end(text)

        if status != "OK":
            out["repair_status"] = status
            return out

        fname = f"{ticker}_{filing_date}_{filing_type}_{accession_number}_USBFALLBACK.txt"
        with open(OUT_DIR / fname, "w", encoding="utf-8") as f:
            f.write(mda_text)

        out["repair_status"] = "OK"
        out["repaired_filename"] = fname

        time.sleep(sleep_seconds)
        return out

    except Exception as e:
        out["repair_status"] = f"ERROR: {str(e)[:120]}"
        return out

In [34]:
usb_df = repair_df.loc[repair_df["ticker"] == "USB"].copy()

print("USB rows:", len(usb_df))
display(usb_df[["ticker", "filing_date", "filing_type", "accession_number"]])

USB rows: 14


,ticker,filing_date,filing_type,accession_number
38,USB,2020-05-07,10-Q,0001193125-20-136359
39,USB,2020-08-06,10-Q,0001193125-20-211979
40,USB,2020-11-05,10-Q,0001193125-20-286983
41,USB,2021-05-04,10-Q,0001193125-21-150169
42,USB,2021-08-03,10-Q,0001193125-21-234982
43,USB,2021-11-02,10-Q,0001193125-21-317037
44,USB,2022-05-03,10-Q,0001193125-22-138788
45,USB,2022-08-04,10-Q,0001193125-22-212040
46,USB,2022-11-01,10-Q,0001193125-22-275036
47,USB,2023-05-08,10-Q,0001193125-23-137980


In [35]:
usb_results = []

for i, row in usb_df.iterrows():
    result = repair_one_usb_fallback(row)
    usb_results.append(result)

    if len(usb_results) % 10 == 0 or len(usb_results) == len(usb_df):
        print(f"Processed {len(usb_results)}/{len(usb_df)}")

usb_log = pd.DataFrame(usb_results)
usb_log.to_csv("/content/usb_fallback_log.csv", index=False)

print("USB fallback repair complete.")
display(usb_log["repair_status"].value_counts())
display(usb_log.head())

Processed 10/14
Processed 14/14
USB fallback repair complete.


,count
repair_status,
NO_END,14


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename
0,USB,2020-05-07,10-Q,0001193125-20-136359,NO_END,d897119d10q.htm,
1,USB,2020-08-06,10-Q,0001193125-20-211979,NO_END,d890129d10q.htm,
2,USB,2020-11-05,10-Q,0001193125-20-286983,NO_END,d947218d10q.htm,
3,USB,2021-05-04,10-Q,0001193125-21-150169,NO_END,d137164d10q.htm,
4,USB,2021-08-03,10-Q,0001193125-21-234982,NO_END,d913559d10q.htm,


In [36]:
def inspect_item_lines_for_row(row):
    idx = get_index_json(row["cik"], row["accession_number"])
    primary_doc = choose_primary_doc(idx, row["filing_type"])
    html = download_filing_doc(row["cik"], row["accession_number"], primary_doc)
    text = html_to_text(html)

    lines = text.splitlines()
    item_lines = []

    for i, line in enumerate(lines):
        clean = re.sub(r"\s+", " ", line).strip()
        low = clean.lower()

        if re.search(r"\bitem\s*[1-9]\b", low) or "management's discussion" in low or "controls and procedures" in low:
            item_lines.append((i, clean))

    print("PRIMARY DOC:", primary_doc)
    print("=" * 120)
    for i, line in item_lines[:200]:
        print(f"{i:05d} | {line}")

    return text

In [37]:
usb_row0 = usb_df.iloc[0]
usb_text_debug = inspect_item_lines_for_row(usb_row0)

PRIMARY DOC: d897119d10q.htm
00257 | 1) Management's Discussion and Analysis of Financial Condition and Results of Operations (Item 2)
00319 | f) Controls and Procedures (Item 4)
00329 | 2) Quantitative and Qualitative Disclosures About Market Risk/Corporate Risk Profile (Item 3)
00439 | 4) Financial Statements (Item 1)
00453 | 1) Legal Proceedings (Item 1)
00473 | 3) Unregistered Sales of Equity Securities and Use of Proceeds (Item 2)
00483 | 4) Exhibits (Item 6)
01981 | Management's Discussion and Analysis
03676 | In addition, credit quality ratings as defined by the Company, are an important part of the Company's overall credit risk management and evaluation of its allowance for credit losses. Loans with a pass rating represent those loans not classified on the Company's rating scale for problem credits, as minimal risk has been identified. Loans with a special mention or classified rating, including loans that are 90 days or more past due and still accruing, nonaccrual loans, those

In [38]:
patterns = [
    r"item\s*3",
    r"item\s*4",
    r"quantitative and qualitative disclosures about market risk",
    r"controls and procedures",
]

for pat in patterns:
    print("\nPATTERN:", pat)
    for m in re.finditer(pat, usb_text_debug, flags=re.I):
        start = max(0, m.start() - 120)
        end = min(len(usb_text_debug), m.start() + 220)
        snippet = usb_text_debug[start:end].replace("\n", " ")
        print(snippet)
        print("-" * 100)
        break


PATTERN: item\s*3
Procedures (Item 4)        32     2) Quantitative and Qualitative Disclosures About Market Risk/Corporate Risk Profile (Item 3)         7     a) Overview        7     b) Credit Risk Management        9     c) Residual Value Risk Management        20     d) Operational Risk Management        20     e) Compliance Risk Management        20  
----------------------------------------------------------------------------------------------------

PATTERN: item\s*4
)  Non-GAAP  Financial Measures        29     e) Critical Accounting Policies        30     f) Controls and Procedures (Item 4)        32     2) Quantitative and Qualitative Disclosures About Market Risk/Corporate Risk Profile (Item 3)         7     a) Overview        7     b) Credit Risk Management        9     c) Residual Value Risk Man
----------------------------------------------------------------------------------------------------

PATTERN: quantitative and qualitative disclosures about market risk
asures   

In [39]:
for pat in [
    r"item\s*2",
    r"management[' ]s discussion and analysis",
    r"management[' ]s discussion and analysis of financial condition and results of operations",
]:
    print("\nPATTERN:", pat)
    for m in re.finditer(pat, usb_text_debug, flags=re.I):
        start = max(0, m.start() - 120)
        end = min(len(usb_text_debug), m.start() + 220)
        snippet = usb_text_debug[start:end].replace("\n", " ")
        print(snippet)
        print("-" * 100)
        break


PATTERN: item\s*2
I — Financial Information     1) Management's Discussion and Analysis of Financial Condition and Results of Operations (Item 2)        3     a) Overview        3     b) Statement of Income Analysis        3     c) Balance Sheet Analysis        6     d)  Non-GAAP  Financial Measures        29     e) Critical Accounting Policies        30  
----------------------------------------------------------------------------------------------------

PATTERN: management[' ]s discussion and analysis
  Table of Contents   Table of Contents and  Form 10-Q  Cross Reference Index     Part I — Financial Information     1) Management's Discussion and Analysis of Financial Condition and Results of Operations (Item 2)        3     a) Overview        3     b) Statement of Income Analysis        3     c) Balance Sheet Analysis        6     d) 
----------------------------------------------------------------------------------------------------

PATTERN: management[' ]s discussion and analys

In [40]:
def extract_usb_real(text):
    text = text.replace("\xa0", " ")
    text = text.replace("’", "'")

    low = text.lower()

    # real start candidates: later MD&A, not TOC
    start_matches = []
    for m in re.finditer(r"management[' ]s discussion and analysis", low, flags=re.I):
        start_matches.append(m.start())

    if not start_matches:
        return None, "NO_START"

    # reject early TOC matches by keeping only later occurrences
    doc_len = len(text)
    start_matches = [s for s in start_matches if s > doc_len * 0.10] or start_matches

    # end candidates based on actual USB text
    end_matches = []
    for pat in [
        r"controls and procedures",
        r"item\s*4[\.\-:\s]+controls and procedures",
        r"^controls and procedures$",
    ]:
        for m in re.finditer(pat, low, flags=re.I | re.M):
            end_matches.append(m.start())

    end_matches = sorted(set(end_matches))

    if not end_matches:
        return None, "NO_END"

    best_pair = None

    for s in start_matches:
        valid_ends = [e for e in end_matches if e > s]
        if not valid_ends:
            continue

        e = valid_ends[0]
        span = e - s

        if 3000 <= span <= 400000:
            best_pair = (s, e)
            break

    if best_pair is None:
        return None, "NO_VALID_SPAN"

    s, e = best_pair
    mda = text[s:e].strip()

    wc = len(re.findall(r"\b[a-zA-Z]+\b", mda))
    if wc < 800:
        return None, "TOO_SHORT"

    first_500 = re.sub(r"\s+", " ", mda[:500]).lower()

    # reject TOC-style starts
    if (
        "table of contents" in first_500 or
        "cross reference index" in first_500 or
        "part i — financial information" in first_500 or
        "item 2)" in first_500
    ):
        return None, "BAD_START"

    # accept later narrative MD&A start
    if "management's discussion and analysis" not in first_500:
        return None, "BAD_START"

    return mda, "OK"

In [41]:
def repair_one_usb_real(row, sleep_seconds=0.2):
    ticker = row["ticker"]
    cik = row["cik"]
    filing_date = row["filing_date"]
    filing_type = row["filing_type"]
    accession_number = row["accession_number"]

    out = {
        "ticker": ticker,
        "filing_date": filing_date,
        "filing_type": filing_type,
        "accession_number": accession_number,
        "repair_status": "",
        "primary_doc": "",
        "repaired_filename": ""
    }

    try:
        if pd.isna(cik):
            out["repair_status"] = "NO_CIK"
            return out

        idx = get_index_json(cik, accession_number)
        primary_doc = choose_primary_doc(idx, filing_type)

        if primary_doc is None:
            out["repair_status"] = "NO_PRIMARY_DOC"
            return out

        out["primary_doc"] = primary_doc

        html = download_filing_doc(cik, accession_number, primary_doc)
        text = html_to_text(html)

        mda_text, status = extract_usb_real(text)

        if status != "OK":
            out["repair_status"] = status
            return out

        fname = f"{ticker}_{filing_date}_{filing_type}_{accession_number}_USBREAL.txt"
        with open(OUT_DIR / fname, "w", encoding="utf-8") as f:
            f.write(mda_text)

        out["repair_status"] = "OK"
        out["repaired_filename"] = fname

        time.sleep(sleep_seconds)
        return out

    except Exception as e:
        out["repair_status"] = f"ERROR: {str(e)[:120]}"
        return out

In [42]:
usb_df = repair_df.loc[repair_df["ticker"] == "USB"].copy()

print("USB rows:", len(usb_df))
display(usb_df[["ticker", "filing_date", "filing_type", "accession_number"]])

USB rows: 14


,ticker,filing_date,filing_type,accession_number
38,USB,2020-05-07,10-Q,0001193125-20-136359
39,USB,2020-08-06,10-Q,0001193125-20-211979
40,USB,2020-11-05,10-Q,0001193125-20-286983
41,USB,2021-05-04,10-Q,0001193125-21-150169
42,USB,2021-08-03,10-Q,0001193125-21-234982
43,USB,2021-11-02,10-Q,0001193125-21-317037
44,USB,2022-05-03,10-Q,0001193125-22-138788
45,USB,2022-08-04,10-Q,0001193125-22-212040
46,USB,2022-11-01,10-Q,0001193125-22-275036
47,USB,2023-05-08,10-Q,0001193125-23-137980


In [43]:
usb_results = []

for i, row in usb_df.iterrows():
    result = repair_one_usb_real(row)
    usb_results.append(result)

    if len(usb_results) % 10 == 0 or len(usb_results) == len(usb_df):
        print(f"Processed {len(usb_results)}/{len(usb_df)}")

usb_log = pd.DataFrame(usb_results)
usb_log.to_csv("/content/usb_real_log.csv", index=False)

print("USB real extraction complete.")
display(usb_log["repair_status"].value_counts())
display(usb_log.head())

Processed 10/14
Processed 14/14
USB real extraction complete.


,count
repair_status,
OK,13
BAD_START,1


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename
0,USB,2020-05-07,10-Q,0001193125-20-136359,OK,d897119d10q.htm,USB_2020-05-07_10-Q_0001193125-20-136359_USBRE...
1,USB,2020-08-06,10-Q,0001193125-20-211979,OK,d890129d10q.htm,USB_2020-08-06_10-Q_0001193125-20-211979_USBRE...
2,USB,2020-11-05,10-Q,0001193125-20-286983,OK,d947218d10q.htm,USB_2020-11-05_10-Q_0001193125-20-286983_USBRE...
3,USB,2021-05-04,10-Q,0001193125-21-150169,OK,d137164d10q.htm,USB_2021-05-04_10-Q_0001193125-21-150169_USBRE...
4,USB,2021-08-03,10-Q,0001193125-21-234982,BAD_START,d913559d10q.htm,


In [44]:
ok_usb = usb_log.loc[usb_log["repair_status"] == "OK"].copy()
print("Recovered OK USB files:", len(ok_usb))
display(ok_usb)

Recovered OK USB files: 13


,ticker,filing_date,filing_type,accession_number,repair_status,primary_doc,repaired_filename
0,USB,2020-05-07,10-Q,0001193125-20-136359,OK,d897119d10q.htm,USB_2020-05-07_10-Q_0001193125-20-136359_USBRE...
1,USB,2020-08-06,10-Q,0001193125-20-211979,OK,d890129d10q.htm,USB_2020-08-06_10-Q_0001193125-20-211979_USBRE...
2,USB,2020-11-05,10-Q,0001193125-20-286983,OK,d947218d10q.htm,USB_2020-11-05_10-Q_0001193125-20-286983_USBRE...
3,USB,2021-05-04,10-Q,0001193125-21-150169,OK,d137164d10q.htm,USB_2021-05-04_10-Q_0001193125-21-150169_USBRE...
5,USB,2021-11-02,10-Q,0001193125-21-317037,OK,d219476d10q.htm,USB_2021-11-02_10-Q_0001193125-21-317037_USBRE...
6,USB,2022-05-03,10-Q,0001193125-22-138788,OK,d306477d10q.htm,USB_2022-05-03_10-Q_0001193125-22-138788_USBRE...
7,USB,2022-08-04,10-Q,0001193125-22-212040,OK,d322057d10q.htm,USB_2022-08-04_10-Q_0001193125-22-212040_USBRE...
8,USB,2022-11-01,10-Q,0001193125-22-275036,OK,d245647d10q.htm,USB_2022-11-01_10-Q_0001193125-22-275036_USBRE...
9,USB,2023-05-08,10-Q,0001193125-23-137980,OK,d796135d10q.htm,USB_2023-05-08_10-Q_0001193125-23-137980_USBRE...
10,USB,2023-08-07,10-Q,0001193125-23-204050,OK,d425808d10q.htm,USB_2023-08-07_10-Q_0001193125-23-204050_USBRE...


In [45]:
for i in range(len(ok_usb)):
    fname = ok_usb.iloc[i]["repaired_filename"]
    print(f"\n### Reviewing USB row {i}")
    preview_repaired_file(fname)


### Reviewing USB row 0
FILE: USB_2020-05-07_10-Q_0001193125-20-136359_USBREAL.txt

--- START PREVIEW ---

Management's Discussion and Analysis — Credit Risk Management” in the Company's Annual Report on 
Form 10-K
 for the year ended December 31, 2019, for a more detailed discussion on credit risk management processes.
 
The Company manages its credit risk, in part, through diversification of its loan portfolio which is achieved through limit setting by product type criteria, such as industry, and identification of credit concentrations. As part of its normal business activities, the Company offers a broad array of lending products. The Company categorizes its loan portfolio into two segments, which is the level at which it develops and documents a systematic methodology to determine the allowance for credit losses. The Company's two loan portfolio segments are commercial lending and consumer lending. 
 
The commercial lending segment includes loans and leases made to small business,

In [46]:
ok_usb = usb_log.loc[
    usb_log["repair_status"] == "OK",
    ["ticker", "filing_date", "filing_type", "accession_number", "repaired_filename"]
].copy()

print("USB OK files:", len(ok_usb))
display(ok_usb)

USB OK files: 13


,ticker,filing_date,filing_type,accession_number,repaired_filename
0,USB,2020-05-07,10-Q,0001193125-20-136359,USB_2020-05-07_10-Q_0001193125-20-136359_USBRE...
1,USB,2020-08-06,10-Q,0001193125-20-211979,USB_2020-08-06_10-Q_0001193125-20-211979_USBRE...
2,USB,2020-11-05,10-Q,0001193125-20-286983,USB_2020-11-05_10-Q_0001193125-20-286983_USBRE...
3,USB,2021-05-04,10-Q,0001193125-21-150169,USB_2021-05-04_10-Q_0001193125-21-150169_USBRE...
5,USB,2021-11-02,10-Q,0001193125-21-317037,USB_2021-11-02_10-Q_0001193125-21-317037_USBRE...
6,USB,2022-05-03,10-Q,0001193125-22-138788,USB_2022-05-03_10-Q_0001193125-22-138788_USBRE...
7,USB,2022-08-04,10-Q,0001193125-22-212040,USB_2022-08-04_10-Q_0001193125-22-212040_USBRE...
8,USB,2022-11-01,10-Q,0001193125-22-275036,USB_2022-11-01_10-Q_0001193125-22-275036_USBRE...
9,USB,2023-05-08,10-Q,0001193125-23-137980,USB_2023-05-08_10-Q_0001193125-23-137980_USBRE...
10,USB,2023-08-07,10-Q,0001193125-23-204050,USB_2023-08-07_10-Q_0001193125-23-204050_USBRE...


In [47]:
import re
import pandas as pd
from pathlib import Path

USB_DIR = OUT_DIR  # this should already point to your USB repaired txt folder

def read_text_safe(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

def word_count_alpha(text):
    return len(re.findall(r"\b[a-zA-Z]+\b", text))

review_rows = []

for _, row in ok_usb.iterrows():
    path = USB_DIR / row["repaired_filename"]
    text = read_text_safe(path)

    review_rows.append({
        "ticker": row["ticker"],
        "filing_date": row["filing_date"],
        "filing_type": row["filing_type"],
        "accession_number": row["accession_number"],
        "repaired_filename": row["repaired_filename"],
        "word_count": word_count_alpha(text),
        "first_300_chars": text[:300].replace("\n", " "),
        "first_1000_chars": text[:1000].replace("\n", " "),
        "last_500_chars": text[-500:].replace("\n", " "),
        "manual_label": "",
        "notes": "",
    })

usb_review_df = pd.DataFrame(review_rows)

print("Rows in USB review CSV:", len(usb_review_df))
display(usb_review_df)

Rows in USB review CSV: 13


,ticker,filing_date,filing_type,accession_number,repaired_filename,word_count,first_300_chars,first_1000_chars,last_500_chars,manual_label,notes
0,USB,2020-05-07,10-Q,0001193125-20-136359,USB_2020-05-07_10-Q_0001193125-20-136359_USBRE...,13649,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,le conclusions that could be significant to th...,,
1,USB,2020-08-06,10-Q,0001193125-20-211979,USB_2020-08-06_10-Q_0001193125-20-211979_USBRE...,14732,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,able conclusions that could be significant to ...,,
2,USB,2020-11-05,10-Q,0001193125-20-286983,USB_2020-11-05_10-Q_0001193125-20-286983_USBRE...,15002,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,le conclusions that could be significant to th...,,
3,USB,2021-05-04,10-Q,0001193125-21-150169,USB_2021-05-04_10-Q_0001193125-21-150169_USBRE...,13613,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,opment and the selection of critical accountin...,,
4,USB,2021-11-02,10-Q,0001193125-21-317037,USB_2021-11-02_10-Q_0001193125-21-317037_USBRE...,14708,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,opment and the selection of critical accountin...,,
5,USB,2022-05-03,10-Q,0001193125-22-138788,USB_2022-05-03_10-Q_0001193125-22-138788_USBRE...,12723,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,opment and the selection of critical accountin...,,
6,USB,2022-08-04,10-Q,0001193125-22-212040,USB_2022-08-04_10-Q_0001193125-22-212040_USBRE...,13271,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,elopment and the selection of critical account...,,
7,USB,2022-11-01,10-Q,0001193125-22-275036,USB_2022-11-01_10-Q_0001193125-22-275036_USBRE...,13556,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,opment and the selection of critical accountin...,,
8,USB,2023-05-08,10-Q,0001193125-23-137980,USB_2023-05-08_10-Q_0001193125-23-137980_USBRE...,12863,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,elopment and the selection of critical account...,,
9,USB,2023-08-07,10-Q,0001193125-23-204050,USB_2023-08-07_10-Q_0001193125-23-204050_USBRE...,13193,Management's Discussion and Analysis — Credit ...,Management's Discussion and Analysis — Credit ...,opment and the selection of critical accountin...,,


In [48]:
USB_REVIEW_CSV = "/content/usb_13_review.csv"
usb_review_df.to_csv(USB_REVIEW_CSV, index=False)

print("Saved:", USB_REVIEW_CSV)

Saved: /content/usb_13_review.csv


In [49]:
from google.colab import files
files.download(USB_REVIEW_CSV)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [50]:
import pandas as pd

correct_2 = pd.DataFrame([
    {
        "ticker": "AAPL",
        "filing_date": "2019-10-31",
        "filing_type": "10-K",
        "accession_number": "0000320193-19-000119",
        "repaired_filename": "AAPL_2019-10-31_10-K_0000320193-19-000119_FAMILY52.txt",
        "source_log": "family_rescue",
        "manual_label": "correct",
        "notes": "real MD&A heading and narrative start"
    },
    {
        "ticker": "KLAC",
        "filing_date": "2023-08-04",
        "filing_type": "10-K",
        "accession_number": "0000319201-23-000031",
        "repaired_filename": "KLAC_2023-08-04_10-K_0000319201-23-000031_FAMILY52.txt",
        "source_log": "family_rescue",
        "manual_label": "correct",
        "notes": "real MD&A heading and narrative start"
    }
])

CORRECT_2_CSV = "/content/correct_2_files.csv"
correct_2.to_csv(CORRECT_2_CSV, index=False)

print("Saved:", CORRECT_2_CSV)
display(correct_2)

Saved: /content/correct_2_files.csv


,ticker,filing_date,filing_type,accession_number,repaired_filename,source_log,manual_label,notes
0,AAPL,2019-10-31,10-K,0000320193-19-000119,AAPL_2019-10-31_10-K_0000320193-19-000119_FAMI...,family_rescue,correct,real MD&A heading and narrative start
1,KLAC,2023-08-04,10-K,0000319201-23-000031,KLAC_2023-08-04_10-K_0000319201-23-000031_FAMI...,family_rescue,correct,real MD&A heading and narrative start


In [51]:
from pathlib import Path
import shutil
import zipfile

SOURCE_DIR = OUT_DIR
DOWNLOAD_DIR = Path("/content/correct_2_txt")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# clear old files
for old_file in DOWNLOAD_DIR.glob("*.txt"):
    old_file.unlink()

copied = 0
missing = []

for _, row in correct_2.iterrows():
    fname = row["repaired_filename"]
    src = SOURCE_DIR / fname
    dst = DOWNLOAD_DIR / fname

    if src.exists():
        shutil.copy2(src, dst)
        copied += 1
    else:
        missing.append(fname)

print("Copied txt files:", copied)
print("Missing txt files:", len(missing))
if missing:
    print("Missing list:")
    for x in missing:
        print(x)

Copied txt files: 2
Missing txt files: 0


In [52]:
CORRECT_2_ZIP = "/content/correct_2_txt.zip"

with zipfile.ZipFile(CORRECT_2_ZIP, "w", zipfile.ZIP_DEFLATED) as zipf:
    for txt_file in DOWNLOAD_DIR.glob("*.txt"):
        zipf.write(txt_file, arcname=txt_file.name)

print("Saved zip:", CORRECT_2_ZIP)

Saved zip: /content/correct_2_txt.zip


In [53]:
from google.colab import files

files.download(CORRECT_2_CSV)
files.download(CORRECT_2_ZIP)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [54]:
remaining_52 = pd.read_csv("remaining_52_files_rebuilt.csv")
correct_2 = pd.read_csv("/content/correct_2_files.csv")

for df in [remaining_52, correct_2]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

correct_2_keys = set(
    zip(
        correct_2["ticker"],
        correct_2["filing_date"],
        correct_2["filing_type"],
        correct_2["accession_number"]
    )
)

remaining_50 = remaining_52.loc[
    ~remaining_52.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in correct_2_keys,
        axis=1
    )
].copy()

REMAINING_50_CSV = "/content/remaining_50_files.csv"
remaining_50.to_csv(REMAINING_50_CSV, index=False)

print("Remaining unresolved files:", len(remaining_50))
print("Saved:", REMAINING_50_CSV)

Remaining unresolved files: 50
Saved: /content/remaining_50_files.csv


In [55]:
remaining_52 = pd.read_csv("remaining_52_files_rebuilt.csv")
correct_2 = pd.read_csv("/content/correct_2_files.csv")

for df in [remaining_52, correct_2]:
    df["ticker"] = df["ticker"].astype(str).str.strip().str.upper()
    df["filing_type"] = df["filing_type"].astype(str).str.strip().str.upper()
    df["accession_number"] = df["accession_number"].astype(str).str.strip()
    df["filing_date"] = pd.to_datetime(df["filing_date"], errors="coerce").dt.strftime("%Y-%m-%d")

correct_2_keys = set(
    zip(
        correct_2["ticker"],
        correct_2["filing_date"],
        correct_2["filing_type"],
        correct_2["accession_number"]
    )
)

remaining_50 = remaining_52.loc[
    ~remaining_52.apply(
        lambda r: (r["ticker"], r["filing_date"], r["filing_type"], r["accession_number"]) in correct_2_keys,
        axis=1
    )
].copy()

REMAINING_50_CSV = "/content/remaining_50_files.csv"
remaining_50.to_csv(REMAINING_50_CSV, index=False)

print("Remaining unresolved files:", len(remaining_50))
print("Saved:", REMAINING_50_CSV)

Remaining unresolved files: 50
Saved: /content/remaining_50_files.csv
